<a href="https://colab.research.google.com/github/Emya101/Personal_RAG_Project/blob/main/pharmaceutical_rag_project_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q llama-index
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-llms-google-genai
!pip install -q chromadb
!pip install -q llama-index-vector-stores-chroma
!pip install -q scikit-learn
!pip install -q nest_asyncio
%pip install llama-index-retrievers-bm25
%pip install pymupdf

In [ ]:
import os
import nest_asyncio
from google.colab import userdata

from llama_index.core import Settings
from llama_index.llms.google_genai import GoogleGenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

nest_asyncio.apply()

# Designate tools

os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")

llm = GoogleGenAI(model = "gemini-3.5-flash-lite")

embed_model = HuggingFaceEmbedding(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from google.colab import files
import fitz
from llama_index.core import Document

# Extract text from pdf with pymupdf
uploaded = files.upload()
pdf_name = next(iter(uploaded))

pdf = fitz.open(pdf_name)

documents = []

for page_number, page in enumerate(pdf, start=1):
    text = page.get_text(sort=True)

    if text.strip():
        documents.append(
            Document(
                text=text,
                metadata={
                    "file_name": pdf_name,
                    "page": page_number
                }
            )
        )

print(f"Extracted text from {len(documents)} pages.")
print(documents[0].text)

Saving sample-sdf-document (Project 5).pdf to sample-sdf-document (Project 5) (4).pdf
Extracted text from 3 pages.
                                                                                                           Cytiva
                                                                                           100 Results Way
                                                                                                Marlborough, MA 01752
                                                                                                          United States

3 June, 2022


Re: ÄKTATM ready Flow Kit Storage Conditions


To Whom It May Concern,

The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits
as well, including the two listed in the below table. Extended storage below the recommended +5   could lea

In [ ]:
from llama_index.core.node_parser import SentenceSplitter
#Implementing chunking(overlapping chunking)

splitter_overlap = SentenceSplitter(chunk_size = 300, chunk_overlap = 50)
chunks_overlap = splitter_overlap.get_nodes_from_documents(documents)
print(f"Total Overlapping Chunks Created: {len(chunks_overlap)}")

print("\nFirst chunk:")
print(chunks_overlap[0].get_content()[:500])

Total Overlapping Chunks Created: 6

First chunk:
Cytiva
                                                                                           100 Results Way
                                                                                                Marlborough, MA 01752
                                                                                                          United States

3 June, 2022


Re: ÄKTATM ready Flow Kit Storage Conditions


To Whom It May Concern,

The recommended storage temperature for standard ÄKTA ready 


In [ ]:

from llama_index.core import VectorStoreIndex

# Implementing Embeddings

index = VectorStoreIndex(
    nodes=chunks_overlap,
    embed_model=embed_model
)

print("Vector index created successfully!")


Vector index created successfully!


In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever


def create_hybrid_retriever(index, query, top_k = 10):
  # Vector Retrieval (Semantic Search)
  vector_retriever = index.as_retriever(
    similarity_top_k = top_k
  )

  vector_nodes = vector_retriever.retrieve(query)

  # Keyword-based Retrieval
  bm25_retriever = BM25Retriever.from_defaults(
    nodes = chunks_overlap,
    similarity_top_k= top_k
  )

  keyword_nodes = bm25_retriever.retrieve(query)

  # Combine Results
  all_nodes = list(vector_nodes) + list(keyword_nodes)

  # Remove duplicates by node_id
  unique_nodes = {}
  for node in all_nodes:
    if node.node_id not in unique_nodes:
      unique_nodes[node.node_id] = node

  # Sort by score (higher is better)
  sorted_nodes = sorted(
      unique_nodes.values(),
      key=lambda x: x.score if hasattr(x, 'score') else 0.0,
        reverse=True
  )

  return sorted_nodes[:top_k]

In [ ]:
# LLM-Assisted Reranking
from llama_index.core.postprocessor import SentenceTransformerRerank

# Create a reranker using a cross-encoder model
reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_n=5
)

def retrieve_context(query):
  # Retrieve top 10 results using Hybrid Retrieval
  retrieved_nodes = create_hybrid_retriever(
      index,
      query,
      top_k = 10
  )

  #Rerank and select the top 5 most relevant results
  reranked_nodes = reranker.postprocess_nodes(
      retrieved_nodes,
      query_str = query
  )

  return reranked_nodes


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [ ]:

def ask_rag(question):

    # Retrieve and rerank relevant document chunks
    retrieved_nodes = retrieve_context(question)

    # Combine retrieved chunks into context
    context = "\n\n".join(
        node.node.get_content() for node in retrieved_nodes
    )

    # Create prompt for Gemini
    prompt = f"""
    You are a pharmaceutical document assistant.

    Answer the question using only the document context below.
    Do not invent information.
    If the information is unavailable, say so.

    Document Context:
    {context}

    Question:
    {question}
    """

    # Generate response using Gemini
    response = llm.complete(prompt)

    return str(response)


In [ ]:
questions = [
    "What test methods were used for quality control?",
    "What are the storage conditions specified in the certificate?"
]

for i, question in enumerate(questions, start = 1):
  answer = ask_rag(question)

  print(f"\nPrompt {i}: {question}")
  print(f"RAG Response:\n{answer}")
  print("-"*60)

DEBUG:bm25s:Building index from IDs objects
DEBUG:bm25s:Building index from IDs objects



Prompt 1: What test methods were used for quality control?
RAG Response:
Based on the provided document, the test methods used are:

* **Flow Rate Test:** Per SOP-QC-042
* **Pressure Integrity:** Max 0.5 bar, 5 min hold
* **Visual Inspection:** No defects visible
* **Package Integrity:** Sealed, no damage
* **Autoclave - Pump tubing:** 121C > 15 min
* **Gamma Irradiation - Inlets:** 25.0 - 40.0 kGy
------------------------------------------------------------



Prompt 2: What are the storage conditions specified in the certificate?
RAG Response:
Based on the provided document, storage conditions are not specified in the certificates (Certificate of Quality). 

However, the document notes under the "ÄKTA ready Flow Kit Storage Conditions" section that:
* The recommended storage temperature for standard and modified ÄKTA ready flow kits is **> +5 °C**.
* Extended storage below the recommended +5 °C could lead to brittleness or cracking of the plastic connectors.
------------------------------------------------------------
